# insurance-governance

**Unified model governance for UK insurance pricing teams.**

This notebook shows the two things UK pricing teams need every time a model goes to committee:
a PRA SS1/23-aligned statistical validation report and a model risk tier assessment.
Both come from a single `pip install`.

The problem: validation tests (Gini, PSI, lift charts) and MRM governance packs (model cards,
risk tier scores) were always built separately and version-pinned separately.
This package merges them. One install, one import path.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/burning-cost/insurance-governance/blob/main/notebooks/quickstart.ipynb)

In [ ]:
!pip install -q insurance-governance numpy

## Synthetic model outputs

We simulate a UK motor frequency model: 5,000 validation policies,
a Poisson-distributed observed claim count, and model-predicted frequencies.
The predictions are calibrated but imperfect — typical of a real model.

In [ ]:
import numpy as np

rng = np.random.default_rng(42)
n_val = 5_000

# Observed: Poisson claim counts on a motor book (mean ~8%)
y_true = rng.poisson(0.08, n_val).astype(float)

# Model predictions: calibrated but with some noise
y_pred = np.clip(rng.normal(0.08, 0.02, n_val), 0.002, None)

# Add some discrimination signal: sorted predictions correlate with outcome
# (realistic model has partial but not perfect lift)
sort_idx = np.argsort(y_pred)
y_true[sort_idx[4000:]] += rng.poisson(0.03, 1000)  # high-pred risks claim more

print(f"Validation set: {n_val:,} policies")
print(f"Observed mean freq: {y_true.mean():.4f}")
print(f"Predicted mean freq: {y_pred.mean():.4f}")
print(f"Calibration ratio: {y_true.mean() / y_pred.mean():.3f}")

## Step 1: Statistical validation (PRA SS1/23)

`ModelValidationReport` runs the battery of tests the PRA expects:
discrimination (Gini), stability (PSI), calibration, and lift.
It produces a RAG-coded summary that goes directly into the Model Validation Pack.

In [ ]:
from insurance_governance import ModelValidationReport, ValidationModelCard

card = ValidationModelCard(
    name="Motor Frequency v3.2",
    version="3.2.0",
    purpose="Predict claim frequency for UK motor portfolio",
    methodology="CatBoost gradient boosting with Poisson objective",
    target="claim_count",
    features=["age", "vehicle_age", "area", "vehicle_group", "ncd_years"],
    limitations=["No telematics data", "Trained on 2020-2023 only"],
    owner="Pricing Team",
)

report = ModelValidationReport(model_card=card, y_val=y_true, y_pred_val=y_pred)
summary = report.summary()

print("Validation summary:")
for test_name, result in summary.items():
    print(f"  {test_name:<30} {result}")

## Step 2: Model Risk Tier scoring

`RiskTierScorer` applies an objective 0-100 composite score across six
dimensions (complexity, data quality, materiality, etc.) and maps it to
Tier 1 / 2 / 3. The tier drives sign-off requirements — Tier 1 needs a full
Model Risk Committee pack, Tier 3 needs an owner attestation only.

In [ ]:
from insurance_governance import MRMModelCard, RiskTierScorer, Assumption, Limitation

mrm_card = MRMModelCard(
    name="Motor Frequency v3.2",
    version="3.2.0",
    model_type="gradient_boosting",
    business_use="Motor personal lines pricing",
    owner="Pricing Team",
    developer="Pricing Analytics",
    assumptions=[
        Assumption("Stationarity", "Claim patterns in 2024+ match 2020-2023 training data"),
        Assumption("Independence", "Policies are independent conditional on rating factors"),
    ],
    limitations=[
        Limitation("Data", "No telematics data integrated", severity="medium"),
        Limitation("Scope", "Excludes commercial and fleet", severity="low"),
    ],
)

scorer = RiskTierScorer()
tier_result = scorer.score(
    model_card=mrm_card,
    annual_premium_gbp=45_000_000,  # £45m book
    n_policyholders=180_000,
    is_regulatory_capital=False,
    uses_ml=True,
)

print(f"Risk tier:    {tier_result.tier}")
print(f"Risk score:   {tier_result.score:.1f} / 100")
print(f"\nDimension scores:")
for dim in tier_result.dimensions:
    print(f"  {dim.name:<25} {dim.score:.0f}/100  —  {dim.rationale}")

## Step 3: Model inventory

`ModelInventory` maintains a persistent JSON registry of all models in production.
It tracks versions, tiers, owner, and last validation date.
This is what the Model Risk Committee looks at in a thematic review.

In [ ]:
import tempfile, os
from insurance_governance import ModelInventory

# Use a temp file — in production this would be a shared drive path
with tempfile.NamedTemporaryFile(suffix=".json", delete=False) as f:
    inventory_path = f.name

inventory = ModelInventory(path=inventory_path)

# Register our model
inventory.register(
    model_card=mrm_card,
    tier_result=tier_result,
    last_validated="2024-11-01",
    next_validation_due="2025-11-01",
    status="production",
)

# Also add a simpler model for comparison
mrm_card2 = MRMModelCard(
    name="Home Frequency v1.5",
    version="1.5.0",
    model_type="glm",
    business_use="Home buildings pricing",
    owner="Pricing Team",
    developer="Pricing Analytics",
    assumptions=[Assumption("Log-link", "Log link appropriate for frequency")],
    limitations=[],
)
tier_result2 = scorer.score(
    model_card=mrm_card2,
    annual_premium_gbp=12_000_000,
    n_policyholders=95_000,
    is_regulatory_capital=False,
    uses_ml=False,
)
inventory.register(
    model_card=mrm_card2,
    tier_result=tier_result2,
    last_validated="2024-06-15",
    next_validation_due="2025-06-15",
    status="production",
)

print("Model inventory:")
print(inventory.to_dataframe())

os.unlink(inventory_path)

## What you should see

- The validation report shows Gini, PSI, and calibration with RAG status.
- Motor Frequency v3.2 (ML, £45m book) should score Tier 1 or Tier 2 — it is
  a large, material, complex model. Home Frequency v1.5 (GLM, £12m) should
  score Tier 3.
- The inventory shows both models with their tier, owner, and next review date.

## Next steps

- **`GovernanceReport`** — generates a full Model Risk Committee pack as Markdown
- **`ReportGenerator`** — exports the validation report as self-contained HTML
- **`DiscriminationReport`** and **`StabilityReport`** — run individual test categories

**GitHub:** https://github.com/burning-cost/insurance-governance  
**PyPI:** https://pypi.org/project/insurance-governance/